Strategia consigliata per l'architettura della tesi

Poiché la tua toolchain prevede un Backbone layer intercambiabile che riceve tensori (B, C, T) e una maschera di condizionamento, conviene strutturare la sperimentazione in modo incrementale:   Baseline veloce (1D U-Net con FiLM): usala per validare l'intera pipeline (training, inversione dello scaling, metriche generative e monitoraggio STL post-hoc con MoonLight o RTAMT).   Confronto architetturale (Transformer fattorizzato stile CSDI o ibrido Conv1D + Attention): introduci un Transformer leggero (2–3 layer, d_model = 64 o 128, con dropout per evitare overfitting) o inserisci un blocco di Self-Attention al bottleneck della 1D U-Net, misurando sui dati di test se il guadagno sulle correlazioni multivariate giustifica il maggior costo computazionale.Vuoi che implementiamo ora la 1D U-Net con condizionamento FiLM oppure preferisci partire direttamente con il Transformer fattorizzato (tempo + feature) stile CSDI compatibile con il Trajectory24hDataset appena creato?

## Import packages

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from einops import rearrange
from diffusers import DDPMScheduler
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from math import sqrt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from keras.models import Sequential, load_model
from keras.layers import Dense, LSTM, Dropout, BatchNormalization
from sklearn.metrics import mean_squared_error as mse
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

CSV_PATH = "datasets/pollution/multivariate_pollution.csv"  # sostituisci con il percorso del tuo file CSV (separatore ',')[cite: 2]

WND_DIR_MAPPING = {
    "NE": 0,
    "SE": 1,
    "NW": 2,
    "cv": 3,
}

FEATURE_COLS = [
    "pollution",
    "dew",
    "temp",
    "press",
    "wnd_dir",
    "wnd_spd",
    "snow",
    "rain",
]

## Data Load

In [ ]:
df_train = pd.read_csv(CSV_PATH)
df_train

In [ ]:
df_train.info()

In [ ]:
# Checking null values
print(df_train.isnull().sum())

In [ ]:
df_train.describe()

In [ ]:
df_test = pd.read_csv('datasets/pollution/pollution_test.csv')
df_test

## Data preprocessing

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

LOG_COLS = ["pollution", "wnd_spd"]

# 1. Copia dei dataframe originali
df_train_clean = df_train.copy()
df_test_clean = df_test.copy()

# 2. Mappatura direzione del vento
df_train_clean["wnd_dir"] = df_train_clean["wnd_dir"].map(WND_DIR_MAPPING)
df_test_clean["wnd_dir"] = df_test_clean["wnd_dir"].map(WND_DIR_MAPPING)

# 3. Indice temporale ed estrazione ora del giorno (0..23)
df_train_clean["date"] = pd.to_datetime(df_train_clean["date"])
df_train_clean.set_index("date", inplace=True)

train_hours = df_train_clean.index.hour.values
# df_train ha 43800 righe (1825 giorni esatti -> finisce alle 23:00), quindi df_test riparte da 00:00
test_hours = np.arange(len(df_test_clean)) % 24


def encode_cyclical_hour(hours_1d: np.ndarray) -> np.ndarray:
    angle = 2.0 * np.pi * hours_1d / 24.0
    return np.stack([np.sin(angle), np.cos(angle)], axis=1).astype(
        np.float32
    )  # (N, 2)


train_hour_cov = encode_cyclical_hour(train_hours)
test_hour_cov = encode_cyclical_hour(test_hours)

# 4. Allineamento colonne
df_train_aligned = df_train_clean[FEATURE_COLS].copy()
df_test_aligned = df_test_clean[FEATURE_COLS].copy()

# 5. Trasformazione log1p sulle feature asimmetriche prima dello scaling
for col in LOG_COLS:
    df_train_aligned[col] = np.log1p(np.clip(df_train_aligned[col], 0.0, None))
    df_test_aligned[col] = np.log1p(np.clip(df_test_aligned[col], 0.0, None))

# 6. Scaling in (-1, 1) fittato SOLO sul train
scaler = MinMaxScaler(feature_range=(-1, 1))
train_scaled_values = scaler.fit_transform(df_train_aligned).astype(np.float32)
test_scaled_values = scaler.transform(df_test_aligned).astype(np.float32)

print("Train scaled shape:", train_scaled_values.shape, "| Cov shape:", train_hour_cov.shape)
print("Test  scaled shape:", test_scaled_values.shape, "| Cov shape:", test_hour_cov.shape)

In [ ]:
df_train_scaled = pd.DataFrame(train_scaled_values, columns=df_train_aligned.columns)

values = df_train_scaled.values
groups = [0, 1, 2, 3, 5, 6]  # 0 è pollution, 1 dew, 2 temp, 3 press, ecc.
i = 1

plt.figure(figsize=(20, 14))
for group in groups:
    plt.subplot(len(groups), 1, i)
    plt.plot(values[:, group], color=cm.plasma(group / len(groups)))
    plt.xlabel("Index")
    plt.title(df_train_scaled.columns[group], y=0.75, loc="right", fontsize=15)
    i += 1
plt.show()

In [ ]:
df_test_scaled = pd.DataFrame(test_scaled_values, columns=df_test_aligned.columns)

sns.set(style="darkgrid")

fig, axs = plt.subplots(3, 2, figsize=(24, 14))

sns.histplot(data=df_test_scaled, x="pollution", kde=True, color="skyblue", ax=axs[0, 0])
sns.histplot(data=df_test_scaled, x="dew", kde=True, color="olive", ax=axs[0, 1])
sns.histplot(data=df_test_scaled, x="temp", kde=True, color="gold", ax=axs[1, 0])
sns.histplot(data=df_test_scaled, x="press", kde=True, color="teal", ax=axs[1, 1])
sns.histplot(data=df_test_scaled, x="wnd_dir", kde=True, color="steelblue", ax=axs[2, 0])
sns.histplot(data=df_test_scaled, x="wnd_spd", kde=True, color="goldenrod", ax=axs[2, 1])

plt.show()

## Dataset with einops

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view
from torch.utils.data import DataLoader, Dataset

WINDOW_SIZE = 24


def make_windows(array_2d: np.ndarray, window_size: int = 24, stride: int = 1) -> np.ndarray:
    windows = sliding_window_view(array_2d, window_shape=window_size, axis=0)[::stride]
    return einops.rearrange(windows, "w c t -> w c t").astype(np.float32)


class Trajectory24hDataset(Dataset):

    def __init__(
            self,
            scaled_array_2d: np.ndarray,
            hour_cov_2d: np.ndarray,
            window_size: int = 24,
    ):
        if len(scaled_array_2d) < window_size:
            self.data = np.empty((0, scaled_array_2d.shape[1], window_size), dtype=np.float32)
            self.cov = np.empty((0, hour_cov_2d.shape[1], window_size), dtype=np.float32)
        else:
            self.data = make_windows(scaled_array_2d, window_size=window_size)
            self.cov = make_windows(hour_cov_2d, window_size=window_size)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.data[idx])  # (C, 24)
        x0 = x[:, 0:1]  # (C, 1)
        cov = torch.from_numpy(self.cov[idx])  # (2, 24)
        return {"x": x, "x0": x0, "cov": cov}


train_dataset = Trajectory24hDataset(train_scaled_values, train_hour_cov, window_size=WINDOW_SIZE)
test_dataset = Trajectory24hDataset(test_scaled_values, test_hour_cov, window_size=WINDOW_SIZE)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

batch = next(iter(train_loader))
print("x:", batch["x"].shape, "| x0:", batch["x0"].shape, "| cov:", batch["cov"].shape)

In [ ]:

import torch
import torch.nn as nn


class SinusoidalTimeEmb(nn.Module):

    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, timesteps: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        freqs = torch.exp(
            -torch.log(torch.tensor(10000.0, device=timesteps.device))
            * torch.arange(half, device=timesteps.device)
            / half
        )
        args = timesteps[:, None].float() * freqs[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class ResBlock1D(nn.Module):

    def __init__(self, in_ch: int, out_ch: int, time_emb_dim: int, groups: int = 8):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, in_ch)
        self.act1 = nn.SiLU()
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=3, padding=1, padding_mode="replicate")

        self.time_proj = nn.Sequential(nn.SiLU(), nn.Linear(time_emb_dim, out_ch))

        self.norm2 = nn.GroupNorm(groups, out_ch)
        self.act2 = nn.SiLU()
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=3, padding=1, padding_mode="replicate")

        self.skip = nn.Conv1d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        h = self.conv1(self.act1(self.norm1(x)))
        h = h + self.time_proj(t_emb)[:, :, None]
        h = self.conv2(self.act2(self.norm2(h)))
        return h + self.skip(x)


class TemporalSelfAttention1D(nn.Module):

    def __init__(self, channels: int, num_heads: int = 4, groups: int = 8):
        super().__init__()
        self.norm = nn.GroupNorm(groups, channels)
        self.attn = nn.MultiheadAttention(embed_dim=channels, num_heads=num_heads, batch_first=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, T)
        h = self.norm(x)
        h_seq = einops.rearrange(h, "b c t -> b t c")
        attn_out, _ = self.attn(h_seq, h_seq, h_seq)
        return x + einops.rearrange(attn_out, "b t c -> b c t")


class UNet1DWithAttention(nn.Module):
    """1D U-Net condizionata su x0, maschera iniziale e covariate orarie (sin/cos)."""

    def __init__(
            self,
            channels: int = 8,
            cov_channels: int = 2,
            base_dim: int = 64,
            time_dim: int = 128,
    ):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmb(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )

        # Ingresso: [x_noisy (C), x0_exp (C), cov (2), cond_mask (1)]
        in_ch = channels * 2 + cov_channels + 1
        self.in_conv = nn.Conv1d(in_ch, base_dim, kernel_size=3, padding=1, padding_mode="replicate")

        # Encoder: T=24 -> T=12 -> T=6
        self.enc1 = ResBlock1D(base_dim, base_dim, time_dim)
        self.down1 = nn.Conv1d(base_dim, base_dim * 2, kernel_size=4, stride=2, padding=1)  # T: 24 -> 12

        self.enc2 = ResBlock1D(base_dim * 2, base_dim * 2, time_dim)
        self.attn2 = TemporalSelfAttention1D(base_dim * 2, num_heads=4)
        self.down2 = nn.Conv1d(base_dim * 2, base_dim * 4, kernel_size=4, stride=2, padding=1)  # T: 12 -> 6

        # Bottleneck: T=6
        self.mid1 = ResBlock1D(base_dim * 4, base_dim * 4, time_dim)
        self.mid_attn = TemporalSelfAttention1D(base_dim * 4, num_heads=8)
        self.mid2 = ResBlock1D(base_dim * 4, base_dim * 4, time_dim)

        # Decoder: T=6 -> T=12 -> T=24
        self.up2 = nn.ConvTranspose1d(base_dim * 4, base_dim * 2, kernel_size=4, stride=2, padding=1)  # T: 6 -> 12
        self.dec2 = ResBlock1D(base_dim * 4, base_dim * 2, time_dim)
        self.dec_attn2 = TemporalSelfAttention1D(base_dim * 2, num_heads=4)

        self.up1 = nn.ConvTranspose1d(base_dim * 2, base_dim, kernel_size=4, stride=2, padding=1)  # T: 12 -> 24
        self.dec1 = ResBlock1D(base_dim * 2, base_dim, time_dim)

        self.out_norm = nn.GroupNorm(8, base_dim)
        self.out_act = nn.SiLU()
        self.out_conv = nn.Conv1d(base_dim, channels, kernel_size=3, padding=1, padding_mode="replicate")

    def forward(
            self,
            x_t: torch.Tensor,
            x0: torch.Tensor,
            cov: torch.Tensor,
            timestep: torch.Tensor,
    ) -> torch.Tensor:
        bsz, _, t_len = x_t.shape
        x0_exp = x0.expand_as(x_t)

        # Maschera esplicita: 1.0 a t=0 (condizione iniziale nota), 0.0 altrove
        cond_mask = torch.zeros((bsz, 1, t_len), device=x_t.device, dtype=x_t.dtype)
        cond_mask[:, :, 0] = 1.0

        h = torch.cat([x_t, x0_exp, cov, cond_mask], dim=1)
        t_emb = self.time_mlp(timestep)

        h0 = self.in_conv(h)
        e1 = self.enc1(h0, t_emb)  # (B, 64, 24)
        e2 = self.attn2(self.enc2(self.down1(e1), t_emb))  # (B, 128, 12)

        m = self.mid1(self.down2(e2), t_emb)  # (B, 256, 6)
        m = self.mid2(self.mid_attn(m), t_emb)  # (B, 256, 6)

        d2 = self.up2(m)  # (B, 128, 12)
        d2 = self.dec_attn2(self.dec2(torch.cat([d2, e2], dim=1), t_emb))

        d1 = self.up1(d2)  # (B, 64, 24)
        d1 = self.dec1(torch.cat([d1, e1], dim=1), t_emb)

        return self.out_conv(self.out_act(self.out_norm(d1)))


class Diffusion24hGenerator(nn.Module):

    def __init__(
            self,
            channels: int = 8,
            cov_channels: int = 2,
            base_dim: int = 64,
            num_train_timesteps: int = 500,
    ):
        super().__init__()
        self.channels = channels
        self.cov_channels = cov_channels
        self.base_dim = base_dim
        self.model = UNet1DWithAttention(
            channels=channels, cov_channels=cov_channels, base_dim=base_dim
        )
        self.scheduler = DDPMScheduler(
            num_train_timesteps=num_train_timesteps, beta_schedule="squaredcos_cap_v2"
        )

    def training_loss(
            self, x: torch.Tensor, x0: torch.Tensor, cov: torch.Tensor
    ) -> torch.Tensor:
        bsz = x.shape[0]
        device = x.device
        timesteps = torch.randint(
            0, self.scheduler.config.num_train_timesteps, (bsz,), device=device
        ).long()
        noise = torch.randn_like(x)

        noisy_x = self.scheduler.add_noise(x, noise, timesteps)
        noisy_x[:, :, 0:1] = x0

        pred_noise = self.model(noisy_x, x0, cov, timesteps)
        return nn.functional.mse_loss(pred_noise[:, :, 1:], noise[:, :, 1:])

    @torch.no_grad()
    def sample(
            self,
            x0: torch.Tensor,
            cov: torch.Tensor,
            window_size: int = 24,
            num_steps: int = 100,
    ) -> torch.Tensor:
        device = x0.device
        bsz, channels, _ = x0.shape

        x_t = torch.randn((bsz, channels, window_size), device=device)
        x_t[:, :, 0:1] = x0

        self.scheduler.set_timesteps(num_steps, device=device)
        for t in self.scheduler.timesteps:
            t_batch = torch.full((bsz,), t, device=device, dtype=torch.long)
            pred_noise = self.model(x_t, x0, cov, t_batch)
            x_t = self.scheduler.step(pred_noise, t, x_t).prev_sample
            x_t[:, :, 0:1] = x0

        return x_t

## Training loop

In [ ]:
from pathlib import Path
import torch
from datetime import datetime

filename = f"best_model_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pt"

CHECKPOINT_DIR = Path("parameters/pollution")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CKPT_PATH = CHECKPOINT_DIR / filename
LAST_CKPT_PATH = CHECKPOINT_DIR / "last_model.pt"


def save_checkpoint(
        path: Path,
        model: Diffusion24hGenerator,
        optimizer: torch.optim.Optimizer,
        epoch: int,
        train_loss: float,
        val_loss: float,
        feature_cols: list[str],
        log_cols: list[str],
        scaler: MinMaxScaler,
):
    torch.save(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "num_train_timesteps": model.scheduler.config.num_train_timesteps,
            "beta_schedule": model.scheduler.config.beta_schedule,
            "window_size": WINDOW_SIZE,
            "channels": model.channels,
            "cov_channels": model.cov_channels,
            "base_dim": model.base_dim,
            "feature_cols": feature_cols,
            "log_cols": log_cols,
            "scaler_min_": scaler.min_,
            "scaler_scale_": scaler.scale_,
            "scaler_data_min_": scaler.data_min_,
            "scaler_data_max_": scaler.data_max_,
            "scaler_feature_range": scaler.feature_range,
        },
        path,
    )


@torch.no_grad()
def evaluate_loss(
        model: Diffusion24hGenerator, loader: DataLoader, device: torch.device
) -> float:
    model.eval()
    total_loss = 0.0
    num_batches = 0
    for batch in loader:
        x = batch["x"].to(device)
        x0 = batch["x0"].to(device)
        cov = batch["cov"].to(device)
        loss = model.training_loss(x, x0, cov)
        total_loss += loss.item()
        num_batches += 1
    return total_loss / max(num_batches, 1)


def train_diffusion_model(
        model: Diffusion24hGenerator,
        train_loader: DataLoader,
        val_loader: DataLoader,
        feature_cols: list[str],
        log_cols: list[str],
        scaler: MinMaxScaler,
        epochs: int = 80,
        lr: float = 5e-4,
        device: torch.device | None = None,
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler_lr = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=lr * 0.02
    )

    best_val_loss = float("inf")
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        num_batches = 0

        for batch in train_loader:
            x = batch["x"].to(device)
            x0 = batch["x0"].to(device)
            cov = batch["cov"].to(device)

            optimizer.zero_grad()
            loss = model.training_loss(x, x0, cov)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item()
            num_batches += 1

        scheduler_lr.step()
        train_loss = running_loss / max(num_batches, 1)
        val_loss = (
            evaluate_loss(model, val_loader, device)
            if len(val_loader) > 0
            else train_loss
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        save_checkpoint(
            LAST_CKPT_PATH,
            model,
            optimizer,
            epoch,
            train_loss,
            val_loss,
            feature_cols,
            log_cols,
            scaler,
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(
                BEST_CKPT_PATH,
                model,
                optimizer,
                epoch,
                train_loss,
                val_loss,
                feature_cols,
                log_cols,
                scaler,
            )
            marker = " <-- best"
        else:
            marker = ""

        if epoch % 5 == 0 or epoch == 1 or marker:
            print(
                f"Epoch {epoch:03d}/{epochs:03d} | "
                f"Train Loss: {train_loss:.5f} | "
                f"Val Loss: {val_loss:.5f}{marker}"
            )

    return history


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = Diffusion24hGenerator(
    channels=len(FEATURE_COLS),
    cov_channels=2,
    base_dim=64,
    num_train_timesteps=500,
)
history = train_diffusion_model(
    model=generator,
    train_loader=train_loader,
    val_loader=test_loader,
    feature_cols=FEATURE_COLS,
    log_cols=LOG_COLS,
    scaler=scaler,
    epochs=80,
    lr=5e-4,
    device=device,
)

## Generate samples from the trained model

In [ ]:
from pathlib import Path
from diffusers import DDPMScheduler
import einops
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import torch


def rebuild_scaler_from_ckpt(ckpt: dict) -> MinMaxScaler:
    sc = MinMaxScaler(feature_range=ckpt["scaler_feature_range"])
    sc.min_ = ckpt["scaler_min_"]
    sc.scale_ = ckpt["scaler_scale_"]
    sc.data_min_ = ckpt["scaler_data_min_"]
    sc.data_max_ = ckpt["scaler_data_max_"]
    sc.n_features_in_ = len(sc.min_)
    return sc


def load_trained_generator(ckpt_path: Path, device: torch.device | None = None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = Diffusion24hGenerator(
        channels=ckpt["channels"],
        cov_channels=ckpt.get("cov_channels", 2),
        base_dim=ckpt.get("base_dim", 64),
        num_train_timesteps=ckpt["num_train_timesteps"],
    )
    model.scheduler = DDPMScheduler(
        num_train_timesteps=ckpt["num_train_timesteps"],
        beta_schedule=ckpt["beta_schedule"],
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    model.eval()

    scaler_loaded = rebuild_scaler_from_ckpt(ckpt)
    return model, scaler_loaded, ckpt


def trajectories_to_original_scale(
        trajectories_bct: torch.Tensor,
        scaler_obj: MinMaxScaler,
        feature_cols: list[str],
        log_cols: list[str],
) -> np.ndarray:
    """Inverte MinMaxScaler(-1, 1) e applica expm1 sulle colonne trasformate con log1p."""
    arr_ntc = einops.rearrange(trajectories_bct.detach().cpu().numpy(), "n c t -> n t c")
    n_traj, t_steps, n_feat = arr_ntc.shape
    inv_2d = scaler_obj.inverse_transform(arr_ntc.reshape(-1, n_feat))
    inv_ntc = inv_2d.reshape(n_traj, t_steps, n_feat)

    for col in log_cols:
        if col in feature_cols:
            idx = feature_cols.index(col)
            inv_ntc[:, :, idx] = np.expm1(np.clip(inv_ntc[:, :, idx], 0.0, 10.0))

    return inv_ntc


@torch.no_grad()
def generate_from_initial_conditions(
        model: Diffusion24hGenerator,
        x0_batch: torch.Tensor,
        cov_batch: torch.Tensor,
        scaler_obj: MinMaxScaler,
        feature_cols: list[str],
        log_cols: list[str],
        num_samples_per_x0: int = 1,
        window_size: int = 24,
        num_steps: int = 100,
        batch_size: int = 128,
        device: torch.device | None = None,
):
    if device is None:
        device = next(model.parameters()).device

    x0_rep = einops.repeat(x0_batch, "m c 1 -> (m k) c 1", k=num_samples_per_x0)
    cov_rep = einops.repeat(cov_batch, "m c t -> (m k) c t", k=num_samples_per_x0)
    all_scaled = []

    for start in range(0, len(x0_rep), batch_size):
        x0_chunk = x0_rep[start: start + batch_size].to(device)
        cov_chunk = cov_rep[start: start + batch_size].to(device)
        gen_chunk = model.sample(
            x0_chunk, cov_chunk, window_size=window_size, num_steps=num_steps
        )
        all_scaled.append(gen_chunk.cpu())

    scaled_bct = torch.cat(all_scaled, dim=0)
    original_ntc = trajectories_to_original_scale(
        scaled_bct, scaler_obj, feature_cols, log_cols
    )
    return scaled_bct, original_ntc


# ==============================================================================
# ESECUZIONE GENERAZIONE & SALVATAGGIO
# ==============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model, loaded_scaler, ckpt_info = load_trained_generator(BEST_CKPT_PATH, device=device)
feature_cols = ckpt_info["feature_cols"]
log_cols = ckpt_info["log_cols"]
window_size = ckpt_info["window_size"]

real_test_bct = torch.from_numpy(test_dataset.data)
real_test_cov = torch.from_numpy(test_dataset.cov)
real_test_ntc = trajectories_to_original_scale(
    real_test_bct, loaded_scaler, feature_cols, log_cols
)

N_COND = min(50, len(real_test_bct))
K_SAMPLES = 10
x0_conditions = real_test_bct[:N_COND, :, 0:1]
cov_conditions = real_test_cov[:N_COND]

gen_scaled_bct, gen_orig_ntc = generate_from_initial_conditions(
    model=loaded_model,
    x0_batch=x0_conditions,
    cov_batch=cov_conditions,
    scaler_obj=loaded_scaler,
    feature_cols=feature_cols,
    log_cols=log_cols,
    num_samples_per_x0=K_SAMPLES,
    window_size=window_size,
    num_steps=100,
    device=device,
)

OUTPUT_DIR = Path("generated_trajectories")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    OUTPUT_DIR / "synthetic_24h.npz",
    gen_orig_ntc=gen_orig_ntc,
    gen_scaled_bct=gen_scaled_bct.numpy(),
    real_ref_ntc=real_test_ntc[:N_COND],
    feature_cols=np.array(feature_cols),
    log_cols=np.array(log_cols),
)

print(f"Generate {gen_orig_ntc.shape[0]} traiettorie da {window_size}h con U-Net + Attention.")

# ==============================================================================
# 3. VISUALIZZAZIONE: ENSEMBLE GENERATO DA UNA SINGOLA CONDIZIONE INIZIALE x0
# ==============================================================================

# Riorganizziamo in (N_COND, K_SAMPLES, T, C) per ispezionare l'ensemble di un singolo x0
ensemble_nk_tc = einops.rearrange(
    gen_orig_ntc, "(m k) t c -> m k t c", m=N_COND, k=K_SAMPLES
)

idx_example = 0  # scegli l'indice della condizione iniziale da ispezionare
vars_to_plot = ["pollution", "temp", "press", "wnd_spd"]
var_indices = [feature_cols.index(v) for v in vars_to_plot if v in feature_cols]

fig, axes = plt.subplots(
    len(var_indices), 1, figsize=(12, 3 * len(var_indices)), sharex=True
)
hours = np.arange(window_size)

for ax, c_idx in zip(axes, var_indices):
    var_name = feature_cols[c_idx]
    real_curve = real_test_ntc[idx_example, :, c_idx]
    sim_curves = ensemble_nk_tc[idx_example, :, :, c_idx]  # (K_SAMPLES, 24)

    mean_curve = sim_curves.mean(axis=0)
    std_curve = sim_curves.std(axis=0)

    # Singole traiettorie campionate
    for k in range(K_SAMPLES):
        ax.plot(
            hours,
            sim_curves[k],
            color="tab:blue",
            alpha=0.22,
            linewidth=1.0,
            label="Campioni generati" if k == 0 else "",
        )

    # Media e banda +/- 1 std dell'ensemble
    ax.plot(
        hours,
        mean_curve,
        color="tab:blue",
        linestyle="--",
        linewidth=2.0,
        label="Media ensemble",
    )
    ax.fill_between(
        hours,
        mean_curve - std_curve,
        mean_curve + std_curve,
        color="tab:blue",
        alpha=0.15,
        label="±1 std",
    )

    # Traiettoria reale osservata a partire dallo stesso x0
    ax.plot(
        hours,
        real_curve,
        color="crimson",
        linewidth=2.2,
        label="Traiettoria reale",
    )
    ax.scatter(
        [0],
        [real_curve[0]],
        color="black",
        zorder=5,
        s=45,
        label="Condizione iniziale x0",
    )

    ax.set_ylabel(var_name)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", ncol=3, fontsize=8)

axes[-1].set_xlabel("Ora (t = 0 ... 23)")
fig.suptitle(
    f"Ensemble di {K_SAMPLES} traiettorie generate da x0 (esempio #{idx_example})",
    fontsize=13,
)
plt.tight_layout()
plt.show()

## Analysis of the generated trajectories

In [ ]:
from scipy.stats import wasserstein_distance
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Carichiamo i dati (funziona sia in continuazione sia in una nuova sessione)
data_npz = np.load(Path("generated_trajectories/synthetic_24h.npz"))
gen_ntc = data_npz["gen_orig_ntc"]
cols = list(data_npz["feature_cols"])
log_cols = list(data_npz["log_cols"])

real_ntc = trajectories_to_original_scale(
    torch.from_numpy(test_dataset.data), loaded_scaler, cols, log_cols
)

# ==============================================================================
# 1. TABELLA STATISTICA & DISTANZA DI WASSERSTEIN (REALE VS GENERATO)
# ==============================================================================

stats_rows = []
for c_idx, col_name in enumerate(cols):
    r_vals = real_ntc[:, :, c_idx].ravel()
    g_vals = gen_ntc[:, :, c_idx].ravel()
    w_dist = wasserstein_distance(r_vals, g_vals)
    stats_rows.append({
        "Feature": col_name,
        "Real Mean": np.mean(r_vals),
        "Gen Mean": np.mean(g_vals),
        "Real Std": np.std(r_vals),
        "Gen Std": np.std(g_vals),
        "Wasserstein 1D": w_dist,
    })

df_stats = pd.DataFrame(stats_rows)
print("=== CONFRONTO STATISTICO MARGINALE (SCALA ORIGINALE) ===")
print(df_stats.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# ==============================================================================
# 2. MONITORAGGIO PROPRIETÀ TEMPORALE (ROBUSTEZZA QUANTITATIVA STILE STL)
# ==============================================================================
# Proprietà esempio: "Esiste un intervallo in cui pollution > soglia per almeno 3 ore consecutive"
# F_[0, 21] ( G_[0, 2] (pollution > threshold) )

poll_idx = cols.index("pollution")
threshold = np.percentile(
    real_ntc[:, :, poll_idx], 75
)  # soglia critica: 75° percentile reale


def stl_persistence_robustness(
        traj_ntc: np.ndarray, var_idx: int, thresh: float, duration: int = 3
) -> np.ndarray:
    """Calcola la robustezza quantitativa per ogni traiettoria (N,):

    rho > 0 <=> proprietà soddisfatta; rho <= 0 <=> violata.
    """
    signal = traj_ntc[:, :, var_idx] - thresh  # (N, 24)
    # Finestra scorrevole di ampiezza `duration`: min interno (operatore Globally), max esterno (Eventually)
    n_traj, t_len = signal.shape
    g_scores = np.stack(
        [
            signal[:, t: t + duration].min(axis=1)
            for t in range(t_len - duration + 1)
        ],
        axis=1,
    )
    return g_scores.max(axis=1)


rho_real = stl_persistence_robustness(real_ntc, poll_idx, threshold, duration=3)
rho_gen = stl_persistence_robustness(gen_ntc, poll_idx, threshold, duration=3)

print(
    f"\n=== MONITORAGGIO PROPRIETÀ TEMPORALE (pollution > {threshold:.1f} per"
    " >= 3h) ==="
)
print(
    f"Tasso soddisfazione Reale:    {(rho_real > 0).mean() * 100:.1f}% |"
    f" Robustezza media: {rho_real.mean():.2f}"
)
print(
    f"Tasso soddisfazione Generato: {(rho_gen > 0).mean() * 100:.1f}% |"
    f" Robustezza media: {rho_gen.mean():.2f}"
)


# ==============================================================================
# 3. TEST DISCRIMINATIVO (CLASSIFICATORE REAL VS SYNTHETIC)
# ==============================================================================


def extract_trajectory_features(arr_ntc: np.ndarray) -> np.ndarray:
    """Estrae feature sintetiche per traiettoria: media, std, min, max, media |delta_t|."""
    mean_f = arr_ntc.mean(axis=1)
    std_f = arr_ntc.std(axis=1)
    min_f = arr_ntc.min(axis=1)
    max_f = arr_ntc.max(axis=1)
    diff_f = np.abs(np.diff(arr_ntc, axis=1)).mean(axis=1)
    return np.concatenate([mean_f, std_f, min_f, max_f, diff_f], axis=1)


n_balanced = min(len(real_ntc), len(gen_ntc))
rng = np.random.default_rng(42)
idx_r = rng.choice(len(real_ntc), size=n_balanced, replace=False)
idx_g = rng.choice(len(gen_ntc), size=n_balanced, replace=False)

X_disc = np.vstack([
    extract_trajectory_features(real_ntc[idx_r]),
    extract_trajectory_features(gen_ntc[idx_g]),
])
y_disc = np.concatenate([np.ones(n_balanced), np.zeros(n_balanced)])

clf = RandomForestClassifier(n_estimators=100, random_state=42)
auc_scores = cross_val_score(clf, X_disc, y_disc, cv=5, scoring="roc_auc")
print("\n=== TEST DISCRIMINATIVO (REAL VS SYNTHETIC) ===")
print(
    f"ROC-AUC (5-fold CV): {auc_scores.mean():.3f} ± {auc_scores.std():.3f} "
    "(0.50 = indistinguibile, 1.00 = facilmente separabile)"
)

# ==============================================================================
# 4. DATA VISUALIZATION DIAGNOSTICA (DISTRIBUZIONI, PROFILO 24H, DINAMICA DELTA)
# ==============================================================================

selected_vars = ["pollution", "temp", "press", "wnd_spd"]
sel_indices = [cols.index(v) for v in selected_vars if v in cols]

fig, axes = plt.subplots(
    len(sel_indices), 3, figsize=(15, 3.2 * len(sel_indices))
)
hours = np.arange(real_ntc.shape[1])

for row, c_idx in enumerate(sel_indices):
    vname = cols[c_idx]
    r_mat = real_ntc[:, :, c_idx]
    g_mat = gen_ntc[:, :, c_idx]

    # Colonna 1: Distribuzione marginale dei valori
    ax_dist = axes[row, 0]
    ax_dist.hist(
        r_mat.ravel(),
        bins=35,
        density=True,
        alpha=0.5,
        color="crimson",
        label="Reale",
    )
    ax_dist.hist(
        g_mat.ravel(),
        bins=35,
        density=True,
        alpha=0.5,
        color="tab:blue",
        label="Generato",
    )
    ax_dist.set_title(f"{vname} — Distribuzione valori")
    ax_dist.set_ylabel("Densità")
    ax_dist.grid(True, alpha=0.25)
    if row == 0:
        ax_dist.legend()

    # Colonna 2: Profilo medio e banda +/- 1 std lungo le 24 ore
    ax_prof = axes[row, 1]
    ax_prof.plot(
        hours, r_mat.mean(axis=0), color="crimson", lw=2, label="Media Reale"
    )
    ax_prof.fill_between(
        hours,
        r_mat.mean(axis=0) - r_mat.std(axis=0),
        r_mat.mean(axis=0) + r_mat.std(axis=0),
        color="crimson",
        alpha=0.15,
    )
    ax_prof.plot(
        hours,
        g_mat.mean(axis=0),
        color="tab:blue",
        lw=2,
        linestyle="--",
        label="Media Gen",
    )
    ax_prof.fill_between(
        hours,
        g_mat.mean(axis=0) - g_mat.std(axis=0),
        g_mat.mean(axis=0) + g_mat.std(axis=0),
        color="tab:blue",
        alpha=0.15,
    )
    ax_prof.set_title(f"{vname} — Profilo temporale (24h)")
    ax_prof.grid(True, alpha=0.25)
    if row == 0:
        ax_prof.legend()

    # Colonna 3: Distribuzione delle variazioni orarie (delta_x = x[t+1] - x[t])
    ax_diff = axes[row, 2]
    r_diff = np.diff(r_mat, axis=1).ravel()
    g_diff = np.diff(g_mat, axis=1).ravel()
    ax_diff.hist(
        r_diff,
        bins=35,
        density=True,
        alpha=0.5,
        color="crimson",
        label="Δx Reale",
    )
    ax_diff.hist(
        g_diff,
        bins=35,
        density=True,
        alpha=0.5,
        color="tab:blue",
        label="Δx Generato",
    )
    ax_diff.set_title(f"{vname} — Incrementi orari Δx")
    ax_diff.grid(True, alpha=0.25)
    if row == 0:
        ax_diff.legend()

axes[-1, 1].set_xlabel("Ora (0 .. 23)")
plt.tight_layout()
plt.show()